# Geometric Brownian Motion with Monte Carlo simulation

해당 노트북에서는 주가의 GBM 운동을 모델링한 확률분포 모델을 이용해서, 몬테카를로 방식을 이용하여 표본을 생성 후 옵션가격을 계산한다.

$$
\ln S_T \sim N\left(\;\ln S_0 + (r-q - \frac{\sigma^2}{2})T , \sigma^2\,T\;\right)\\[0.5em]
S_T^{(i)} = S_0\exp{\left((r-q-\frac{\sigma^2}{2})T + \sigma \sqrt{T}Z_i\right)} \qquad Z_i \sim N(0,1)
$$

마지막 식은 Ito process 로부터 바로 유도된다.
$$
\ln \left( \frac{S_T}{S_0} \right) = \left( r-q-\frac{\sigma^2}{2} \right)\,T + \sigma\,W_T \qquad W_T = \sqrt{T}Z_i
$$

Hull, Chapter 14 에서 유도한 주가의 로그정규분포를 시뮬레이션으로 모델링한다. 그리고, 해당 확률분포에서 몬테카를로 샘플링을 한다.
샘플링한 값들을 이용해서, 

1. 각 표본 $S_T^{(i)}$에서 경로별 만기 payoff를 계산한다. 예를 들어 call은 $H_i=\max(S_T^{(i)}-K,0)$이다.
2. 경로별 payoff를 무위험이자율 $r$로 현재까지 할인하고, 그 표본평균으로 위험중립 기대값을 추정한다. 경로별 payoff와 payoff의 기댓값은 서로 구분한다.
3. 
    - monte carlo result
    - standard error
    - 95% confidence interval
    등의 정보를 출력한다.

## 통계 계산

1. 경로별 할인 payoff 계산하기

Call :
$$
Y_i = e^{-rT}\max{(S_T^{(i)}-K,0)}
$$
Put : 
$$
Y_i = e^{-rT}\max{(K-S_T^{(i)},0)}
$$

2. Monte Carlo 가격 =  표본 평균

$$
\hat{V}_M = \frac{1}{M}\sum_{i=1}^{M}Y_i
$$

3. 할인 payoff의 표본표준편차 (`ddof=1`)
$$
s_Y = \sqrt{ \frac{1}{M-1}\sum_{i=1}^{M}(Y_i-\hat{V}_M)^2 }
$$

4. standard error of sample avg
$$
SE(\hat{V}_M) = \frac{s_Y}{\sqrt{M}}
$$

5. 95% confidence interval

중심극한정리에 의하면, 충분히 큰 경로수에 대해서
$$
\hat{V}_M \pm 1.96\,SE(\hat{V}_M)
$$
값 범위를 사용해서 Call/Put price의 범위를 추정한다. 신뢰구간은 유한 표본에서 참값을 반드시 포함한다는 뜻이 아니라, 반복 표본추출에 대한 정규근사 구간이다.

In [ ]:
# package 함수 import
import pandas as pd

from option_pricing_volatility.simulation.monte_carlo import mc_price

In [ ]:
# 기존 Binomial 노트북과 동일한 synthetic call/put 예제
synthetic_parameters = {
    "spot": 100.0,
    "strike": 100.0,
    "maturity": 1.0,
    "rate": 0.05,
    "volatility": 0.2,
    "dividend_yield": 0.0,
}
example_n_paths = 100_000
example_seed = 42

example_rows = []
for option_type in ("call", "put"):
    result = mc_price(
        **synthetic_parameters,
        n_paths=example_n_paths,
        option_type=option_type,
        seed=example_seed,
    )
    example_rows.append(
        {
            "option_type": option_type,
            "n_paths": result.n_paths,
            "seed": result.seed,
            "mc_price": result.price,
            "standard_error": result.standard_error,
            "ci_low": result.ci_low,
            "ci_high": result.ci_high,
        }
    )

example_results = pd.DataFrame(
    example_rows,
    columns=[
        "option_type",
        "n_paths",
        "seed",
        "mc_price",
        "standard_error",
        "ci_low",
        "ci_high",
    ],
)
example_results

## 수렴 실험

경로 수가 증가하면 Monte Carlo 표준오차는 장기적으로 $1/\sqrt{M}$ 규모로 감소하지만, 개별 난수 표본 때문에 추정가격이나 향후 계산할 BSM 대비 절대오차가 경로 수마다 단조롭게 감소한다고 가정하지 않는다. 아래 tidy-form 결과에는 후속 작업에서 `bsm_price`, `absolute_error`, `relative_error` 열을 바로 추가할 수 있다. 이번 단계에서는 BSM 가격과 최종 비교 그래프를 계산하지 않는다.

In [ ]:
# 여러 경로 수에 대한 수렴 실험용 tidy 결과 테이블
n_paths_grid = [1_000, 5_000, 10_000, 50_000, 100_000]
convergence_seed = 42

convergence_rows = []
for option_type in ("call", "put"):
    for n_paths in n_paths_grid:
        result = mc_price(
            **synthetic_parameters,
            n_paths=n_paths,
            option_type=option_type,
            seed=convergence_seed,
        )
        convergence_rows.append(
            {
                "option_type": option_type,
                "n_paths": result.n_paths,
                "seed": result.seed,
                "mc_price": result.price,
                "standard_error": result.standard_error,
                "ci_low": result.ci_low,
                "ci_high": result.ci_high,
            }
        )

convergence_results = pd.DataFrame(
    convergence_rows,
    columns=[
        "option_type",
        "n_paths",
        "seed",
        "mc_price",
        "standard_error",
        "ci_low",
        "ci_high",
    ],
)

# 향후 bsm_price, absolute_error, relative_error 열을 추가한다.
convergence_results